In [49]:
# Install required packages
!pip install rank-bm25 sentence-transformers groq google-generativeai

In [50]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import google.generativeai as genai
from groq import Groq

# Configure APIs
genai.configure(api_key="YOUR_GEMINI_API_KEY")
groq_client = Groq(api_key="YOUR_GROQ_API_KEY")

In [51]:
corpus = [
    # Attention & Transformers
    "The attention mechanism allows transformers to weigh the importance of each token relative to others, capturing contextual meaning across the sequence.",
    "Multi-head attention runs several attention functions in parallel, letting the model jointly attend to information from different representation subspaces.",
    "Positional encodings are added to token embeddings in transformers because self-attention is permutation-invariant and has no built-in sense of order.",
    "The scaled dot-product attention formula is: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V, where scaling prevents vanishing gradients in deep models.",

    # Neural Network Training
    "Backpropagation computes gradients by applying the chain rule layer by layer, enabling efficient weight updates throughout the network.",
    "Batch normalization normalizes activations within a mini-batch, stabilizing training and allowing higher learning rates.",
    "Dropout is a regularization technique that randomly zeroes out neurons during training, reducing overfitting by preventing co-adaptation.",

    # Optimization
    "Stochastic gradient descent (SGD) updates model weights using gradients computed on a small random batch rather than the full dataset.",
    "The Adam optimizer combines momentum and adaptive learning rates using first and second moment estimates of the gradient.",
    "Learning rate scheduling adjusts the step size during training — common strategies include cosine annealing and warmup followed by decay.",
    "Gradient clipping caps the norm of gradients during backpropagation to prevent exploding gradients in recurrent neural networks.",

    # Embeddings & Representation
    "Word2Vec learns dense word representations by predicting surrounding context words (skip-gram) or the center word from context (CBOW).",
    "Sentence-BERT (SBERT) fine-tunes BERT with siamese networks to produce semantically meaningful sentence embeddings for similarity tasks.",

    # Technical jargon doc — BM25 friendly
    "LoRA (Low-Rank Adaptation) inserts trainable rank-decomposition matrices into frozen transformer layers, enabling parameter-efficient fine-tuning of LLMs.",
]

In [52]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k  # RRF constant

        # BM25 index (tokenize on whitespace, lowercase)
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        # SBERT index
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # --- BM25 ranking ---
        bm25_scores = self.bm25.get_scores(query.lower().split())
        # argsort descending → rank 1 = best
        bm25_ranked = np.argsort(bm25_scores)[::-1].tolist()
        bm25_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

        # --- SBERT ranking ---
        query_emb = self.sbert.encode([query], convert_to_numpy=True)
        # Cosine similarity
        norms = np.linalg.norm(self.doc_embeddings, axis=1) * np.linalg.norm(query_emb)
        sbert_scores = (self.doc_embeddings @ query_emb.T).flatten() / norms
        sbert_ranked = np.argsort(sbert_scores)[::-1].tolist()
        sbert_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

        # --- RRF fusion ---
        rrf_scores = {}
        for doc_id in range(len(self.corpus)):
            r_bm25 = bm25_rank_map[doc_id]
            r_sbert = sbert_rank_map[doc_id]
            rrf_scores[doc_id] = (
                1 / (self.k + r_bm25) +
                1 / (self.k + r_sbert)
            )

        # Sort by RRF score descending
        ranked_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)

        results = []
        for doc_id in ranked_ids[:top_k]:
            results.append({
                "doc_id":     doc_id,
                "rrf_score":  round(rrf_scores[doc_id], 6),
                "bm25_rank":  bm25_rank_map[doc_id],
                "sbert_rank": sbert_rank_map[doc_id],
                "text":       self.corpus[doc_id],
            })
        return results

In [53]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """Re-rank candidates using a cross-encoder. Query must be the original user query."""
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)  # can be negative — that's normal

    for doc, score in zip(candidates, scores):
        doc["cross_encoder_score"] = float(score)

    reranked = sorted(candidates, key=lambda x: x["cross_encoder_score"], reverse=True)
    return reranked[:top_k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [54]:
def hyde_expand(user_query: str) -> str:
    prompt = (
        f"Write a concise, factual, 2-3 sentence technical explanation that directly "
        f"answers the following question. Use precise academic language.\n\n"
        f"Question: {user_query}"
    )
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",  # updated model
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    hypothetical_doc = response.choices[0].message.content.strip()
    print(f"[HyDE] Hypothetical doc: {hypothetical_doc}\n")
    return hypothetical_doc

In [55]:
# Initialize retriever once
retriever = HybridRetriever(corpus, k=60)

def naive_rag(user_query: str) -> str:
    """
    Naïve RAG: dense-only SBERT retrieval, no expansion, no re-ranking.
    Used as the baseline for comparison.
    """
    query_emb = retriever.sbert.encode([user_query], convert_to_numpy=True)
    norms = np.linalg.norm(retriever.doc_embeddings, axis=1) * np.linalg.norm(query_emb)
    scores = (retriever.doc_embeddings @ query_emb.T).flatten() / norms
    top_ids = np.argsort(scores)[::-1][:3]
    context = "\n".join([f"- {corpus[i]}" for i in top_ids])

    prompt = f"Answer the question using the context below.\n\nContext:\n{context}\n\nQuestion: {user_query}"
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


def advanced_rag(user_query: str) -> str:
    """
    Full pipeline: Query Expansion (HyDE) → Hybrid Retrieval → Re-Ranking → LLM Generation
    Returns the final answer string.
    """
    # Step 1 — HyDE query expansion
    expanded_query = hyde_expand(user_query)

    # Step 2 — Hybrid retrieval using the expanded query
    candidates = retriever.retrieve(expanded_query, top_k=7)

    # Step 3 — Re-rank using the ORIGINAL user query (not HyDE)
    top_docs = rerank(user_query, candidates, top_k=3)

    # Step 4 — Build context and generate answer
    context = "\n".join([f"- {doc['text']}" for doc in top_docs])

    prompt = (
        f"You are a university knowledge assistant. Answer the student's question "
        f"clearly and concisely using the context provided.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {user_query}"
    )
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [56]:
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is parameter-efficient fine-tuning?",   # your own query — targets the LoRA doc
]

def get_naive_top_doc(query):
    query_emb = retriever.sbert.encode([query], convert_to_numpy=True)
    norms = np.linalg.norm(retriever.doc_embeddings, axis=1) * np.linalg.norm(query_emb)
    scores = (retriever.doc_embeddings @ query_emb.T).flatten() / norms
    top_id = int(np.argmax(scores))
    return corpus[top_id]

def get_advanced_top_doc(query):
    expanded = hyde_expand(query)
    candidates = retriever.retrieve(expanded, top_k=7)
    reranked = rerank(query, candidates, top_k=1)
    return reranked[0]["text"]

print("Running comparison...\n")
results = []
for q in test_queries:
    print(f"Query: {q}")
    naive_top  = get_naive_top_doc(q)
    adv_top    = get_advanced_top_doc(q)
    different  = "✅ Yes" if naive_top != adv_top else "❌ No"
    results.append((q, naive_top, adv_top, different))
    print(f"  Naïve top doc:    {naive_top[:80]}...")
    print(f"  Advanced top doc: {adv_top[:80]}...")
    print(f"  Different: {different}\n")

Running comparison...

Query: how do transformers encode meaning?
[HyDE] Hypothetical doc: Transformers encode meaning through a self-attention mechanism, where the model attends to specific input elements and their interactions to compute weighted sums of their representations. This process is facilitated by the encoder's multi-head attention layers, which generate a set of attention weights that capture the relevance of each input element to the task at hand. The resulting weighted sums are then combined to produce a contextualized representation of the input sequence.

  Naïve top doc:    Positional encodings are added to token embeddings in transformers because self-...
  Advanced top doc: Positional encodings are added to token embeddings in transformers because self-...
  Different: ❌ No

Query: optimization techniques for training
[HyDE] Hypothetical doc: Optimization techniques for training machine learning models involve the use of algorithms to minimize the loss function and 

## Part 6 — Comparison Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|---|---|---|---|
| `how do transformers encode meaning?` | Positional encodings are added to token embeddings in transformers because self-attention is permutation-invariant... | Positional encodings are added to token embeddings in transformers because self-attention is permutation-invariant... | ❌ No |
| `optimization techniques for training` | The Adam optimizer combines momentum and adaptive learning rates using first and second moment estimates... | Dropout is a regularization technique that randomly zeroes out neurons during training, reducing overfitting... | ✅ Yes |
| `what is parameter-efficient fine-tuning?` | LoRA (Low-Rank Adaptation) inserts trainable rank-decomposition matrices into frozen transformer layers... | LoRA (Low-Rank Adaptation) inserts trainable rank-decomposition matrices into frozen transformer layers... | ❌ No |

## Observations

- **Query 2** shows the pipeline diverging: Naïve RAG picks Adam (direct keyword match on "optimization"),
  while Advanced RAG surfaces Dropout — the HyDE expansion mentioned regularization, which pushed
  Dropout higher via BM25, and the cross-encoder then ranked it above Adam for this query.
- **Queries 1 and 3** return the same top doc in both pipelines, which is expected — when the correct
  document is strongly dominant in dense retrieval, expansion and re-ranking confirm rather than change it.
- The LoRA document is a good example of BM25 strength: the HyDE expansion used terms like
  "subset of parameters" and "pre-trained model", which helped BM25 surface LoRA even though
  the query itself didn't contain those exact words.
- Overall, Advanced RAG adds the most value on ambiguous or semantically broad queries where
  dense retrieval alone may latch onto the wrong similarity signal.

In [57]:
def weighted_rrf(query: str, alpha: float, top_k: int = 5, k: int = 60) -> list[dict]:
    """RRF with configurable weight alpha for BM25, (1-alpha) for SBERT."""

    # BM25 ranking
    bm25_scores = retriever.bm25.get_scores(query.lower().split())
    bm25_ranked = np.argsort(bm25_scores)[::-1].tolist()
    bm25_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

    # SBERT ranking
    query_emb = retriever.sbert.encode([query], convert_to_numpy=True)
    norms = np.linalg.norm(retriever.doc_embeddings, axis=1) * np.linalg.norm(query_emb)
    sbert_scores = (retriever.doc_embeddings @ query_emb.T).flatten() / norms
    sbert_ranked = np.argsort(sbert_scores)[::-1].tolist()
    sbert_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

    # Weighted RRF
    rrf_scores = {}
    for doc_id in range(len(corpus)):
        r_bm25  = bm25_rank_map[doc_id]
        r_sbert = sbert_rank_map[doc_id]
        rrf_scores[doc_id] = (
            alpha       * (1 / (k + r_bm25)) +
            (1 - alpha) * (1 / (k + r_sbert))
        )

    ranked_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return [{"doc_id": i, "rrf_score": round(rrf_scores[i], 6), "text": corpus[i]} for i in ranked_ids[:top_k]]


# Experiment with alpha values
alphas = [0.3, 0.5, 0.7]

# keyword-heavy query vs semantic query
test_cases = {
    "keyword-heavy": "LoRA Low-Rank Adaptation fine-tuning",
    "semantic":      "how do neural networks learn from data?"
}

for label, query in test_cases.items():
    print(f"\n=== Query type: {label} ===")
    print(f"Query: {query}\n")
    for alpha in alphas:
        results = weighted_rrf(query, alpha=alpha, top_k=1)
        print(f"  alpha={alpha} (BM25 weight={alpha}, SBERT weight={1-alpha})")
        print(f"  Top doc: {results[0]['text'][:90]}...")
        print(f"  RRF score: {results[0]['rrf_score']}\n")


=== Query type: keyword-heavy ===
Query: LoRA Low-Rank Adaptation fine-tuning

  alpha=0.3 (BM25 weight=0.3, SBERT weight=0.7)
  Top doc: LoRA (Low-Rank Adaptation) inserts trainable rank-decomposition matrices into frozen trans...
  RRF score: 0.016393

  alpha=0.5 (BM25 weight=0.5, SBERT weight=0.5)
  Top doc: LoRA (Low-Rank Adaptation) inserts trainable rank-decomposition matrices into frozen trans...
  RRF score: 0.016393

  alpha=0.7 (BM25 weight=0.7, SBERT weight=0.30000000000000004)
  Top doc: LoRA (Low-Rank Adaptation) inserts trainable rank-decomposition matrices into frozen trans...
  RRF score: 0.016393


=== Query type: semantic ===
Query: how do neural networks learn from data?

  alpha=0.3 (BM25 weight=0.3, SBERT weight=0.7)
  Top doc: Gradient clipping caps the norm of gradients during backpropagation to prevent exploding g...
  RRF score: 0.01595

  alpha=0.5 (BM25 weight=0.5, SBERT weight=0.5)
  Top doc: Gradient clipping caps the norm of gradients during backpropagat

In [58]:
# A longer document to chunk
long_document = """
Transformers are a type of deep learning model that rely on self-attention mechanisms
to process sequential data. Unlike recurrent neural networks, transformers process all
tokens in parallel, making them highly efficient on modern hardware. The architecture
consists of an encoder and a decoder, each made up of multiple layers of multi-head
attention and feed-forward networks. Positional encodings are added to token embeddings
to preserve order information. The attention mechanism computes query, key, and value
matrices from the input, and uses scaled dot-product attention to weigh token interactions.
Multi-head attention allows the model to attend to different parts of the sequence
simultaneously. Layer normalization and residual connections are applied after each
sub-layer to stabilize training. The feed-forward network applies two linear transformations
with a ReLU activation in between. During training, transformers use the Adam optimizer
with a warmup schedule. The encoder produces contextual embeddings while the decoder
generates output tokens autoregressively. BERT uses only the encoder for classification
tasks, while GPT uses only the decoder for generation tasks.
"""

def chunk_document(text: str, chunk_size: int) -> list[str]:
    """Split text into chunks of approximately chunk_size words."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks


query = "how does multi-head attention work?"

print(f"Query: {query}\n")
for chunk_size in [50, 100, 200]:
    chunks = chunk_document(long_document, chunk_size)

    # Build a mini BM25 index on just these chunks
    tokenized = [c.lower().split() for c in chunks]
    mini_bm25 = BM25Okapi(tokenized)

    # Build a mini SBERT index
    chunk_embs = retriever.sbert.encode(chunks, convert_to_numpy=True)
    query_emb  = retriever.sbert.encode([query], convert_to_numpy=True)
    norms      = np.linalg.norm(chunk_embs, axis=1) * np.linalg.norm(query_emb)
    sbert_scores = (chunk_embs @ query_emb.T).flatten() / norms
    top_sbert  = int(np.argmax(sbert_scores))

    bm25_scores = mini_bm25.get_scores(query.lower().split())
    top_bm25    = int(np.argmax(bm25_scores))

    print(f"--- Chunk size: {chunk_size} words -> {len(chunks)} chunks ---")

Query: how does multi-head attention work?

--- Chunk size: 50 words -> 4 chunks ---
--- Chunk size: 100 words -> 2 chunks ---
--- Chunk size: 200 words -> 1 chunks ---


In [59]:
def colbert_maxsim(query_embs: np.ndarray, doc_embs: np.ndarray) -> float:
    """ColBERT MaxSim: for each query token, find max similarity to any doc token."""
    scores = query_embs @ doc_embs.T  # shape: (q_tokens, d_tokens)
    return float(scores.max(axis=1).mean())  # mean of per-token max scores


class ColBERTRetriever:
    def __init__(self, corpus: list[str]):
        self.corpus = corpus
        self.model  = SentenceTransformer("all-MiniLM-L6-v2")
        # Encode each doc as token-level embeddings (one embedding per word, approximate)
        self.doc_token_embs = []
        for doc in corpus:
            # Encode each word separately as a proxy for token embeddings
            words = doc.split()
            embs  = self.model.encode(words, convert_to_numpy=True)
            self.doc_token_embs.append(embs)

    def score(self, query: str) -> list[float]:
        query_words = query.split()
        query_embs  = self.model.encode(query_words, convert_to_numpy=True)
        scores = []
        for doc_embs in self.doc_token_embs:
            scores.append(colbert_maxsim(query_embs, doc_embs))
        return scores


# Initialize ColBERT retriever
colbert_retriever = ColBERTRetriever(corpus)

def hybrid_retrieve_three(query: str, top_k: int = 5, k: int = 60) -> list[dict]:
    """Three-way hybrid: BM25 + SBERT + ColBERT fused with RRF."""

    # BM25
    bm25_scores  = retriever.bm25.get_scores(query.lower().split())
    bm25_ranked  = np.argsort(bm25_scores)[::-1].tolist()
    bm25_rank_map = {d: r + 1 for r, d in enumerate(bm25_ranked)}

    # SBERT
    query_emb    = retriever.sbert.encode([query], convert_to_numpy=True)
    norms        = np.linalg.norm(retriever.doc_embeddings, axis=1) * np.linalg.norm(query_emb)
    sbert_scores = (retriever.doc_embeddings @ query_emb.T).flatten() / norms
    sbert_ranked = np.argsort(sbert_scores)[::-1].tolist()
    sbert_rank_map = {d: r + 1 for r, d in enumerate(sbert_ranked)}

    # ColBERT
    colbert_scores = colbert_retriever.score(query)
    colbert_ranked = np.argsort(colbert_scores)[::-1].tolist()
    colbert_rank_map = {d: r + 1 for r, d in enumerate(colbert_ranked)}

    # Three-way RRF
    rrf_scores = {}
    for doc_id in range(len(corpus)):
        rrf_scores[doc_id] = (
            1 / (k + bm25_rank_map[doc_id]) +
            1 / (k + sbert_rank_map[doc_id]) +
            1 / (k + colbert_rank_map[doc_id])
        )

    ranked_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return [
        {
            "doc_id":       i,
            "rrf_score":    round(rrf_scores[i], 6),
            "bm25_rank":    bm25_rank_map[i],
            "sbert_rank":   sbert_rank_map[i],
            "colbert_rank": colbert_rank_map[i],
            "text":         corpus[i],
        }
        for i in ranked_ids[:top_k]
    ]

# Test it
print("=== Three-way Hybrid (BM25 + SBERT + ColBERT) ===\n")
for q in test_queries:
    results = hybrid_retrieve_three(q, top_k=3)
    print(f"Query: {q}")
    for r in results:
        print(f"  [{r['rrf_score']}] BM25={r['bm25_rank']} SBERT={r['sbert_rank']} ColBERT={r['colbert_rank']} | {r['text'][:80]}...")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Three-way Hybrid (BM25 + SBERT + ColBERT) ===

Query: how do transformers encode meaning?
  [0.04918] BM25=1 SBERT=1 ColBERT=1 | Positional encodings are added to token embeddings in transformers because self-...
  [0.048387] BM25=2 SBERT=2 ColBERT=2 | The attention mechanism allows transformers to weigh the importance of each toke...
  [0.047131] BM25=5 SBERT=3 ColBERT=3 | LoRA (Low-Rank Adaptation) inserts trainable rank-decomposition matrices into fr...

Query: optimization techniques for training
  [0.047875] BM25=3 SBERT=2 ColBERT=3 | Learning rate scheduling adjusts the step size during training — common strategi...
  [0.047627] BM25=2 SBERT=3 ColBERT=4 | Batch normalization normalizes activations within a mini-batch, stabilizing trai...
  [0.047448] BM25=7 SBERT=1 ColBERT=2 | The Adam optimizer combines momentum and adaptive learning rates using first and...

Query: what is parameter-efficient fine-tuning?
  [0.04918] BM25=1 SBERT=1 ColBERT=1 | LoRA (Low-Rank Adaptation) ins